# Adaptive, corrective, and agentic RAG

**Track:** Enterprise Knowledge Assistant · **Stage:** Production patterns

A production RAG system should not use the same path for every question. Some questions need no retrieval. Some need exact lexical matching. Some need graph traversal. Some need corrective recovery after weak retrieval. This notebook implements a bounded routing and corrective loop.

## What you will build

- A deterministic implementation that runs without API keys.
- A visible trace of evidence, decisions, and failure modes.
- A production design note explaining how this maps to real RAG libraries and systems.

## Concept map

```mermaid
flowchart TD
  Q["Question"] --> Router["Route"]
  Router --> V["Vector / semantic"]
  Router --> H["Hybrid"]
  Router --> G["Graph"]
  V --> Grade["Evidence sufficient?"]
  H --> Grade
  G --> Grade
  Grade -->|"yes"| Answer["Answer with citations"]
  Grade -->|"no"| Correct["Rewrite / alternate route"]
  Correct --> Limit["Retry budget"]
  Limit --> Answer
  Limit --> Abstain["Abstain"]
```

## Setup

Run this notebook from the repository root, or open it in GitHub and copy cells into a local Jupyter session. The helper code lives in `src/enterprise_rag` so the notebook remains readable while the implementation stays testable.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT))

def show(obj):
    print(json.dumps(obj, indent=2))

We run three questions: an exact operational code, a graph/multi-hop question, and an unsupported cafeteria question. The last one should not be forced into a confident answer.

In [ ]:
from src.enterprise_rag.lab_experiments import build_enterprise_chunks, adaptive_trace
chunks = build_enterprise_chunks(ROOT / "data/enterprise")
for q in ["What does AX-774-B mean?", "Who supplies Project Atlas technology and what regulation applies?", "What is NovaTech cafeteria menu today?"]:
    print("\nQUESTION:", q)
    show(adaptive_trace(q, chunks))

### Agentic RAG boundary

Agentic retrieval can plan searches, inspect evidence, and decide when enough context exists. It should not bypass authorization, retry forever, use hidden tools, or treat retrieved instructions as system instructions. Retrieved content is data.

## Deliberate failure case

Before moving on, make the system fail on purpose. Change one variable: chunk size, query wording, top-k, reranking terms, route choice, or evaluation labels. Write down whether the failure belongs to ingestion, retrieval, evidence selection, generation, authorization, or operations.

In [ ]:
# Try your own failure experiment here.
# Example: lower top_k to 1, ask an unsupported question, or remove an important query term.
from src.enterprise_rag.lab_experiments import build_enterprise_chunks
question = "What policy covers parental leave?"
chunks = build_enterprise_chunks(ROOT / "data/enterprise")
print("Question:", question)
print("Now change the query, top_k, or chunking strategy and rerun a comparison helper.")

## Reflection questions

1. What did the simplest baseline get right?
2. What failure was invisible until you inspected the trace?
3. Which component would you improve first in production, and how would you prove it helped?
4. What should the system do when evidence is missing, unauthorized, stale, or contradictory?

## References and next reading

- Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*: https://arxiv.org/abs/2005.11401
- Stanford IR book: https://nlp.stanford.edu/IR-book/
- LangChain retrieval concepts: https://docs.langchain.com/oss/python/langchain/retrieval
- LlamaIndex RAG guide: https://docs.llamaindex.ai/en/stable/understanding/rag/
- Haystack pipeline docs: https://docs.haystack.deepset.ai/docs/pipelines
- Ragas metrics: https://docs.ragas.io/en/stable/concepts/metrics/